# MiMoSA ComparativeStats Example

This notebook combines statistics exported by multiple independent MiMoSA tracking runs while preserving the identity of every run, strain, and cell. MiMoSA is based on upstream RABiTPy and expands its analysis workflow with provenance-aware cross-run comparisons.

It demonstrates validation, table comparisons, run-and-tumble plots, fitted speed comparisons, and translational and angular MSD comparisons.

The notebook is intentionally unexecuted because its source run exports are not distributed with this repository. Configure the run folders below before executing the cells in order.


In [ ]:
# Import the comparison API and notebook helpers
from pathlib import Path

from IPython.display import display
from MiMoSA import ComparativeStats

%matplotlib inline


## 1. Configure Run Folders


Each tuple in `runs` contains `(run_folder, strain, dataset_name)`.
`run_folder` may be a complete tracking-run folder or its `02_Outputs` folder.
Use a unique `dataset_name` for every independent run, and add or remove tuples
to match the datasets being compared. The placeholder folders below are not
included in this repository.


In [ ]:
# Replace this placeholder with the parent of your exported run folders
runs_root = Path('../your_exported_runs').expanduser()

# Generated tables and plots are kept outside version control
output_dir = Path('02_Outputs')

runs = [
    (runs_root / 'Strain_A_Run_1', 'Strain_A', 'Strain_A_run_1'),
    (runs_root / 'Strain_A_Run_2', 'Strain_A', 'Strain_A_run_2'),
    (runs_root / 'Strain_B_Run_1', 'Strain_B', 'Strain_B_run_1'),
    (runs_root / 'Strain_B_Run_2', 'Strain_B', 'Strain_B_run_2'),
]

missing_run_folders = [
    str(folder)
    for folder, _, _ in runs
    if not folder.is_dir()
]
if missing_run_folders:
    raise FileNotFoundError(
        'Replace the example run folders before continuing. Missing folders: '
        f'{missing_run_folders}'
    )

output_dir.mkdir(parents=True, exist_ok=True)


## 2. Load and Validate Runs


In [ ]:
# Load every independent run once and preserve its run identity
comparison = ComparativeStats()

for run_folder, strain, dataset_name in runs:
    comparison.load_dataframes(
        run_folder,
        strain,
        dataset_name=dataset_name,
    )

dataset_registry = comparison.get_dataset_registry()
validation_report = comparison.get_validation_report()

dataset_registry.to_csv(output_dir / 'Dataset_Registry.csv', index=False)
validation_report.to_csv(output_dir / 'Validation_Report.csv', index=False)

display(dataset_registry)
display(validation_report)

## 3. Compare Turn and Run-and-Tumble Tables


### 3.1 Turn Analysis


In [ ]:
turn_df = comparison.turn_analysis_comparison(
    export_csv_path=output_dir,
)
display(turn_df)

### 3.2 Angle-Based Run-and-Tumble Analysis


In [ ]:
# Compare angle-based summaries using equal-particle population tables
angle_df = comparison.angle_tumble_analysis_comparison(
    export_csv_path=output_dir,
)
display(angle_df)


### 3.3 Velocity-Based Run-and-Tumble Analysis


In [ ]:
# Compare velocity-based summaries using equal-particle population tables
velocity_df = comparison.velocity_tumble_analysis_comparison(
    export_csv_path=output_dir,
)
display(velocity_df)


## 4. Plot Comparisons


### 4.1 Plot Settings


In [ ]:
# Set common plotting parameters
comparison.set_analysis_plotting_parameters(
    title_fontsize=16,
    x_axis_fontsize=13,
    y_axis_fontsize=13,
    font_family='DejaVu Sans',
    dpi=300,
)


### 4.2 Fitted Mean-Speed Distribution Comparison


In [ ]:
# Plot the fitted mean speed distribution comparison
speed_summary, speed_figure, speed_axis = (
    comparison.fitted_mean_speed_distribution_comparison(
        show_histogram=True,
        bins=30,
        export_csv_path=output_dir,
        save_plot_path=output_dir / 'Fitted_Speed_Comparison.png',
    )
)
display(speed_summary)


### 4.3 Run-and-Tumble Comparisons


In [ ]:
# Plot run-and-tumble comparisons based on the Najafi-style panels
run_tumble_figure, run_tumble_axes = comparison.plot_run_and_tumble_comparison(
    split_figures=True,
    panel_b_curve=None,  # Use 'normal' to add the fitted speed curve
    panel_b_curve_resolution=300,
    panel_e_curve='normal',  # Also accepts 'binned', 'kde', or None
    panel_e_curve_resolution=500,
    figsize=(17, 10),
    legend_position='middle bottom',
    save_path=output_dir / 'Run_Tumble_Comparison.png',
)

run_tumble_csv_paths = comparison.export_run_and_tumble_comparison_data(
    output_dir / 'Run_Tumble_Data',
)


### 4.4 Translational MSD Comparison


In [ ]:
# Plot and export the translational MSD comparison
msd_figure, msd_axis = comparison.plot_msd_comparison(
    save_path=output_dir / 'MSD_Comparison.png',
)
msd_csv_path = comparison.export_msd_comparison(output_dir)


### 4.5 Angular MSD Comparison


In [ ]:
# Plot the Najafi Figure 1-style angular MSD comparison
amsd_figure, amsd_axis = comparison.plot_angular_msd_comparison(
    figsize=(7, 5),
    legend_position='middle bottom',
    save_path=output_dir / 'Angular_MSD_Comparison.png',
)
